# RAG Pipeline with IBM Granite and LangChain 🚀🤖

<a href="https://colab.research.google.com/github/YOUR_USERNAME/YOUR_REPO/blob/main/RAG_Pipeline_IBM_Granite_LangChain.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

This notebook builds a Retrieval-Augmented Generation (RAG) pipeline that grounds an LLM's
answers in a real source document, instead of letting it hallucinate.

It uses the **State of the Union address** as the knowledge base, **IBM Granite** (via
Replicate) as the LLM, and a local **Milvus Lite** vector store for retrieval.

**What's different from the original version of this notebook:** token handling is now safe
against Colab runtime restarts, and there's a validation step that fails fast with a clear
error message instead of a confusing `401` deep into the pipeline.

## 1. Install dependencies

In [20]:
! echo "::group::Install Dependencies"
%pip install -q uv python-dotenv
! uv pip install --system -q git+https://github.com/ibm-granite-community/utils.git \
    transformers \
    langchain_classic \
    langchain_community \
    langchain_text_splitters \
    langchain_huggingface sentence_transformers \
    langchain_milvus 'pymilvus[milvus_lite]' \
    'langchain_replicate @ git+https://github.com/ibm-granite-community/langchain-replicate.git' \
    replicate \
    wget
! echo "::endgroup::"

::group::Install Dependencies
::endgroup::


## 2. Set up your Replicate API token

**Why this section changed:** the original notebook used `getpass` to store the token only in
`os.environ`. If your Colab runtime restarts (idle timeout, crash, "Restart session"), that
value disappears silently, and any cell run afterward fails with a missing-variable or `401`
error that looks unrelated to the real cause.

This version gives you two options — pick whichever fits how you're running the notebook.

In [21]:
# OPTION A (recommended in Colab): use Colab's built-in Secrets manager.
# Click the key icon 🔑 in the left sidebar, add a secret named REPLICATE_API_TOKEN,
# paste your token as the value, and toggle "Notebook access" on.
# This persists across runtime restarts, unlike plain getpass.

import os

try:
    from google.colab import userdata
    os.environ["REPLICATE_API_TOKEN"] = userdata.get("REPLICATE_API_TOKEN").strip()
    print("Loaded REPLICATE_API_TOKEN from Colab Secrets.")
except Exception:
    print("Colab Secrets not available or not set — falling back to manual entry below.")

Loaded REPLICATE_API_TOKEN from Colab Secrets.


In [22]:
# OPTION B (manual fallback / local Jupyter): paste it directly.
# Run this cell ONLY if Option A above printed the fallback message.

import os, getpass

if not os.environ.get("REPLICATE_API_TOKEN"):
    os.environ["REPLICATE_API_TOKEN"] = getpass.getpass("Paste your Replicate token:").strip()

print("Token is set!" if os.environ.get("REPLICATE_API_TOKEN") else "Token is MISSING")

Token is set!


In [23]:
# Validate the token BEFORE building anything else.
# This is the key fix: catch a bad/expired/scope-limited token here,
# with a clear message, instead of a mysterious 401 later in the pipeline.

import replicate

def validate_token(token: str) -> None:
    try:
        client = replicate.Client(api_token=token)
        list(client.models.list())
    except Exception as e:
        raise SystemExit(
            "ERROR: Your Replicate token was rejected (401/403).\n"
            f"Underlying error: {e}\n\n"
            "Things to check:\n"
            "  1. No extra whitespace/newline in the token (this notebook already strips it).\n"
            "  2. Token wasn't revoked — generate a new one at "
            "https://replicate.com/account/api-tokens\n"
            "  3. Your Replicate account has billing enabled if the model requires it.\n"
        )

validate_token(os.environ["REPLICATE_API_TOKEN"])
print("Replicate token validated successfully.")

Replicate token validated successfully.


## 3. Choose your embeddings model

Specify the model used to generate embedding vectors from text.

In [24]:
from langchain_huggingface import HuggingFaceEmbeddings
from transformers import AutoTokenizer

embeddings_model_path = "ibm-granite/granite-embedding-30m-english"
embeddings_model = HuggingFaceEmbeddings(model_name=embeddings_model_path)
embeddings_tokenizer = AutoTokenizer.from_pretrained(embeddings_model_path)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## 4. Choose your vector database

We use Milvus Lite — a local, file-based version of Milvus that needs no separate server.

In [25]:
from langchain_milvus import Milvus
import tempfile
import uuid

db_file = f"{tempfile.gettempdir()}/milvus_{uuid.uuid4().hex}.db"
print(f"The vector database will be saved to {db_file}")

vector_db = Milvus(
    embedding_function=embeddings_model,
    connection_args={"uri": db_file},
    auto_id=True,
  index_params={"index_type": "AUTOINDEX", "metric_type": "L2"},
)

The vector database will be saved to /tmp/milvus_e368d87f5472452d9608ff3b79918e13.db


## 5. Choose your LLM

The LLM answers the question, given the retrieved context. We use IBM Granite
via the Replicate API.

In [26]:
from langchain_replicate import ChatReplicate

model_path = "ibm-granite/granite-4.0-h-small"

model = ChatReplicate(
    model=model_path,
    replicate_api_token=os.environ["REPLICATE_API_TOKEN"],
)

## 6. Building the vector database

Download the State of the Union address, split it into chunks, embed each chunk,
and load it into the vector database.

In [27]:
import os as _os
import wget

filename = "state_of_the_union.txt"
url = "https://raw.githubusercontent.com/IBM/watson-machine-learning-samples/master/cloud/data/foundation_models/state_of_the_union.txt"

if not _os.path.isfile(filename):
    wget.download(url, out=filename)

In [28]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter

loader = TextLoader(filename)
documents = loader.load()

text_splitter = CharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer=embeddings_tokenizer,
    chunk_size=embeddings_tokenizer.max_len_single_sentence,
    chunk_overlap=0,
)
texts = text_splitter.split_documents(documents)

doc_id = 0
for text in texts:
    doc_id += 1
    text.metadata["doc_id"] = doc_id

print(f"{len(texts)} text document chunks created")

19 text document chunks created


In [29]:
# NOTE: this can take over a minute depending on your embedding model and service.
ids = vector_db.add_documents(texts)
print(f"{len(ids)} documents added to the vector database")

ERROR:grpc._server:Exception calling application: Method not implemented!
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/grpc/_server.py", line 608, in _call_behavior
    response_or_iterator = behavior(argument, context)
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pymilvus/grpc_gen/milvus_pb2_grpc.py", line 1264, in AllocTimestamp
    raise NotImplementedError('Method not implemented!')
NotImplementedError: Method not implemented!


19 documents added to the vector database


## 7. Querying the vector database

Search the database for chunks that are semantically similar to a query.

In [30]:
query = "What did the president say about Ketanji Brown Jackson?"
docs = vector_db.similarity_search(query)
print(f"{len(docs)} documents returned")
for doc in docs:
    print(doc)
    print("=" * 80)

4 documents returned
page_content='Tonight. I call on the Senate to: Pass the Freedom to Vote Act. Pass the John Lewis Voting Rights Act. And while you’re at it, pass the Disclose Act so Americans can know who is funding our elections. 

Tonight, I’d like to honor someone who has dedicated his life to serve this country: Justice Stephen Breyer—an Army veteran, Constitutional scholar, and retiring Justice of the United States Supreme Court. Justice Breyer, thank you for your service. 

One of the most serious constitutional responsibilities a President has is nominating someone to serve on the United States Supreme Court. 

And I did that 4 days ago, when I nominated Circuit Court of Appeals Judge Ketanji Brown Jackson. One of our nation’s top legal minds, who will continue Justice Breyer’s legacy of excellence. 

A former top litigator in private practice. A former federal public defender. And from a family of public school educators and police officers. A consensus builder. Since she’

## 8. Answering questions: build the RAG chain

We create a prompt template that forces the model to answer **only** from the
retrieved context, then wire it together with the retriever.

In [31]:
from langchain_core.prompts import ChatPromptTemplate

template = """Answer the question based ONLY on the following context:
{context}

Question: {input}
"""

prompt_template = ChatPromptTemplate.from_template(template)

In [32]:
from ibm_granite_community.langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain

combine_docs_chain = create_stuff_documents_chain(
    llm=model,
    prompt=prompt_template,
)

rag_chain = create_retrieval_chain(
    retriever=vector_db.as_retriever(),
    combine_docs_chain=combine_docs_chain,
)

## 9. Generate a retrieval-augmented answer

In [33]:
from ibm_granite_community.notebook_utils import wrap_text

query = "What did the president say about Ketanji Brown Jackson?"

output = rag_chain.invoke({"input": query})

print(wrap_text(output["answer"]))

The president described Ketanji Brown Jackson as one of the nation's top legal
minds, a former top litigator in private practice, a former federal public
defender, and a consensus builder. He also mentioned that she has received a
broad range of support since her nomination.


---
## Troubleshooting

**`401 Unauthorized` from Replicate**
- Token has trailing whitespace — this notebook strips it automatically, but check for stray quotes if you're using Colab Secrets.
- Token was revoked — generate a fresh one at replicate.com/account/api-tokens.
- Your account lacks access/billing for this specific Granite model — check the model page while logged in.

**Missing `REPLICATE_API_TOKEN` after it worked before**
- The Colab runtime restarted and wiped `os.environ`. Re-run the token-setup cells (Option A/B above) — using Colab Secrets avoids this entirely going forward.
